# PySpark — Schemas: StructType & StructField

A **schema** defines the column names, data types, and nullability of a DataFrame.  
PySpark can infer schemas automatically, but **explicit schemas are always preferred in production**.

| Approach | When to use |
|----------|-------------|
| `inferSchema=True` | Quick exploration / prototyping |
| Explicit `StructType` | Production — guarantees types, prevents surprises |

Why explicit schemas?
- Avoid an extra scan of the file (infer requires reading data)
- Catch type mismatches early
- Self-documenting code — schema is the contract between systems
- Required when creating DataFrames from Python data

## 1. Inferred Schema — Quick and Easy

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, ArrayType, MapType
)
from pyspark.sql.functions import col

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("Schemas") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    ("Alice",   "Engineering", 95000, True,  3.5),
    ("Bob",     "Marketing",   72000, False, 4.1),
    ("Charlie", "Engineering", 110000, True, 5.0),
]
df_inferred = spark.createDataFrame(data, ["name", "dept", "salary", "active", "rating"])
df_inferred.printSchema()
df_inferred.show()

## 2. Explicit Schema — StructType + StructField

```python
StructField(name, dataType, nullable=True)
```
- `nullable=True` — column can contain `null` values
- `nullable=False` — column must always have a value

In [ ]:
# Explicit schema — guarantees types
schema = StructType([
    StructField("name",     StringType(),  nullable=False),
    StructField("dept",     StringType(),  nullable=True),
    StructField("salary",   IntegerType(), nullable=False),
    StructField("active",   BooleanType(), nullable=True),
    StructField("rating",   DoubleType(),  nullable=True),
])

df_explicit = spark.createDataFrame(data, schema)
df_explicit.printSchema()
df_explicit.show()

# Check column types programmatically
print("Salary type:", df_explicit.schema["salary"].dataType)  # IntegerType()
print("Nullable?  :", df_explicit.schema["name"].nullable)    # False

## 3. Common PySpark Data Types

| PySpark Type | Python equivalent | SQL equivalent |
|---|---|---|
| `StringType()` | `str` | VARCHAR / STRING |
| `IntegerType()` | `int` (32-bit) | INT |
| `LongType()` | `int` (64-bit) | BIGINT |
| `FloatType()` | `float` (32-bit) | FLOAT |
| `DoubleType()` | `float` (64-bit) | DOUBLE |
| `BooleanType()` | `bool` | BOOLEAN |
| `DateType()` | `datetime.date` | DATE |
| `TimestampType()` | `datetime.datetime` | TIMESTAMP |
| `ArrayType(elemType)` | `list` | ARRAY |
| `MapType(keyType, valType)` | `dict` | MAP |

In [ ]:
from datetime import date, datetime

# Schema with dates, timestamps, nested types
rich_schema = StructType([
    StructField("emp_id",     LongType(),       nullable=False),
    StructField("name",       StringType(),     nullable=False),
    StructField("salary",     DoubleType(),     nullable=True),
    StructField("hire_date",  DateType(),       nullable=True),
    StructField("skills",     ArrayType(StringType()), nullable=True),
    StructField("meta",       MapType(StringType(), StringType()), nullable=True),
])

rich_data = [
    (1, "Alice", 95000.0, date(2020, 1, 15), ["Python", "SQL"],    {"level": "senior", "office": "NYC"}),
    (2, "Bob",   72000.0, date(2022, 6, 1),  ["Java", "PySpark"],  {"level": "mid",    "office": "LA"}),
]

df_rich = spark.createDataFrame(rich_data, rich_schema)
df_rich.printSchema()
df_rich.show(truncate=False)

## 4. Schema from a DDL String (Quick Alternative)

You can define a schema as a SQL DDL string instead of StructType — shorter for simple schemas.

In [ ]:
# DDL string schema — same as StructType but shorter
ddl_schema = "name STRING NOT NULL, dept STRING, salary INT, active BOOLEAN, rating DOUBLE"

df_ddl = spark.createDataFrame(data, ddl_schema)
df_ddl.printSchema()
df_ddl.show()

## 5. Reading CSV with Explicit Schema

Always provide schema when reading production data — avoids the extra scan and type guessing.

In [ ]:
# Define schema before reading
emp_schema = StructType([
    StructField("emp_id",  IntegerType(), nullable=False),
    StructField("name",    StringType(),  nullable=False),
    StructField("dept",    StringType(),  nullable=True),
    StructField("salary",  IntegerType(), nullable=True),
    StructField("active",  BooleanType(), nullable=True),
])

# Usage with CSV
# df = spark.read \
#     .schema(emp_schema) \
#     .option("header", "true") \
#     .csv("employees.csv")

# With inferSchema (slower — reads file twice)
# df = spark.read.option("header", "true").option("inferSchema", "true").csv("employees.csv")

print("Schema defined — ready to use with spark.read.schema(emp_schema).csv(...)")
print(emp_schema.simpleString())

## 6. Inspecting and Modifying Schemas

In [ ]:
# printSchema — tree view
df_explicit.printSchema()

# dtypes — list of (name, type_string) tuples
print("dtypes:", df_explicit.dtypes)

# schema — StructType object
print("schema:", df_explicit.schema)

# Check a specific field
print("salary type:", df_explicit.schema["salary"].dataType)

# Cast a column to a different type
df_cast = df_explicit.withColumn("salary", col("salary").cast("double"))
df_cast.printSchema()

# Get column names only
print("columns:", df_explicit.columns)

## Quick Reference

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

# Define schema
schema = StructType([
    StructField("name",   StringType(),  nullable=False),
    StructField("age",    IntegerType(), nullable=True),
    StructField("salary", DoubleType(),  nullable=True),
])

# Create DataFrame with schema
df = spark.createDataFrame(data, schema)

# Read CSV with schema
df = spark.read.schema(schema).option("header", True).csv("file.csv")

# Inspect schema
df.printSchema()          # tree view
df.dtypes                 # list of (col, type) tuples
df.schema["col"].dataType # type of a specific column

# Cast a column
df.withColumn("age", col("age").cast("integer"))
```

## Interview Questions

1. What is the difference between `inferSchema=True` and an explicit schema?
2. What is `StructType` and `StructField`?
3. Why is `nullable=False` important in a schema?
4. What is the difference between `IntegerType()` and `LongType()`?
5. How do you read a CSV with a schema you define yourself?
6. What does `printSchema()` show that `dtypes` does not?
7. How would you handle a CSV column that arrives as String but needs to be treated as Integer?

## Practice Exercises

**Easy:**
1. Create a DataFrame for 3 products (id: int, name: string, price: double, in_stock: boolean) using an explicit schema.
2. Call `printSchema()` and `dtypes` on it — what's the difference in output?

**Medium:**
3. Define a schema for an order: order_id (long), customer (string), amount (double), order_date (date), items (array of strings).
4. Read the Cabs.csv file using an explicit schema (pick 4-5 columns, define their types).

**Advanced:**
5. Create a schema with a nested `StructType` for an address field: street, city, zip.
6. Read a file with `inferSchema`, then convert the schema to an explicit `StructType` definition and verify they match.

In [ ]:
spark.stop()